In [1]:
import numpy as np
import os, sys 
import ROOT
from cats.cdataframe import CDataFrame
import glob

import matplotlib.pyplot as plt
%matplotlib inline

CDMS = os.environ["CDMS"] # set in .bash_profile
stylesheet = os.path.join(CDMS,"scripts","stylesheets","blues.mplstyle")
plt.style.use(stylesheet)

sys.path.append(os.path.join(os.path.join(CDMS,"scripts")))
import setup

Welcome to JupyROOT 6.28/10


In [2]:
path = '/scratch/group/mitchcomp/CDMS/data/perry5334/SourceSimOutput_decayAncestor_Isotope/'
testFile = path + 'CUTE_Cf252_test_23250117_000000.root'

In [3]:
mczipFrame = CDataFrame(f"G4SimDir/mczip1", [testFile])

In [6]:
mczipFrame.AsNumpy(['decayAncestor_PName'])

{'decayAncestor_PName': ndarray([' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '], dtype='<U1')}

In [7]:
mczipFrame.Snapshot("mczip1", path + 'savedFrame.root', ['decayAncestor_PName'])

<cppyy.gbl.ROOT.RDF.RResultPtr<ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager,void> > object at 0x557ed7e52570>

In [8]:
print(CDataFrame(f"mczip1", [path + 'savedFrame.root']).AsNumpy())

{'decayAncestor_PName': ndarray([' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' ', ' '], dtype='<U1')}


In [21]:
mczipFrame.AsNumpy()

runtime_error: const vector<ROOT::VecOps::RVec<char> >& ROOT::RDF::RResultPtr<vector<ROOT::VecOps::RVec<char> > >::GetValue() =>
    runtime_error: An error was encountered while processing the data. TTreeReader status code is: 6

Error in <TTreeReaderArrayBase::GetBranchContentDataType()>: The branch decayAncestor_PName was created using a leaf list and cannot be represented as a C++ type. Please access one of its siblings using a TTreeReaderArray:
Error in <TTreeReaderArrayBase::GetBranchContentDataType()>:    decayAncestor_PName.decayAncestor.PName
Error in <TTreeReaderArrayBase::CreateContentProxy()>: Cannot determine the type contained in the collection of branch decayAncestor_PName. That's weird - please report!


In [2]:
def createFilter_EvtNum_DetNum():

	ROOT.gInterpreter.Declare("""
	#include <vector>
	#include <algorithm>

	bool eventDetFilter(const double colEventNum, const std::vector<double>& targetEventNums, const double colDetNum, const double DetNum) {
		bool eventMatch = std::find(targetEventNums.begin(), targetEventNums.end(), colEventNum) != targetEventNums.end();
		return eventMatch && (DetNum == colDetNum);
	}
	""")


def createFilter_EvtNum():

	ROOT.gInterpreter.Declare("""
	#include <vector>
	#include <algorithm>

	bool eventFilter(const double colEventNum, const std::vector<double>& targetEventNums) {
		bool eventMatch = std::find(targetEventNums.begin(), targetEventNums.end(), colEventNum) != targetEventNums.end();
		return eventMatch;
	}
	""")

createFilter_EvtNum_DetNum()
createFilter_EvtNum()

In [3]:
path = '/scratch/group/mitchcomp/CDMS/data/perry5334/SourceSimOutput_decayAncestor_Isotope/'
DMCfiles = np.sort(glob.glob(path + f'CUTE_Cf252_????????_??????.root'))[10:11]

In [4]:
det = 1
mczipFrame = CDataFrame(f"G4SimDir/mczip{det}", DMCfiles)
mcDecaysFrame = CDataFrame("G4SimDir/mcDecays", DMCfiles)

# Save array of events where neutron capture and Ge71 activation occurred. Determined by recoil/decay of Ga71 nucleus.
GeActivEvents = np.unique(mczipFrame.Filter('string(PName.data()) == "Ga71"').AsNumpy(['EventNum'])['EventNum'])
targetEventNums = ROOT.std.vector("double")(GeActivEvents)

In [5]:
GeActivEventsStr = '{ ' + ', '.join([str(i) for i in GeActivEvents]) + ' }'

In [6]:
mczipFrame = CDataFrame(f"G4SimDir/mczip{det}", DMCfiles)
mcDecaysFrame = CDataFrame(f"G4SimDir/mcDecays", DMCfiles)
mcFluxCounterFrame = CDataFrame(f"G4SimDir/mcFluxCounter", DMCfiles)

mczipFrame_filtered = mczipFrame.Filter("eventFilter(EventNum, " + GeActivEventsStr + ")")
mcDecaysFrame_filtered = mcDecaysFrame.Filter("eventDetFilter(EventNum, " + GeActivEventsStr + f", DetNum, {det})")
mcFluxCounterFrame_filtered = mcFluxCounterFrame.Filter("eventFilter(EventNum, " + GeActivEventsStr + ")")

In [7]:
branches = ['EventNum', 'PName', 'KE', 'Edep', 'Time1', 'Time3', 'decayAncestor.PName', 'decayAncestor.Process', 
            'decayAncestor.T', 'decayAncestor.VolName']

In [8]:
mczip = mczipFrame_filtered.AsNumpy(branches)
mcDecays = mcDecaysFrame_filtered.AsNumpy(branches)
mcFluxCounter = mcFluxCounterFrame_filtered.AsNumpy(branches)

In [10]:
type(mczip['PName'])

ROOT._pythonization._rdf_utils.ndarray

In [15]:
type(np.array(mczip['PName']))

numpy.ndarray

In [17]:
import uproot

# Create a ROOT file to store the tree
with uproot.recreate('output_uproot.root') as file:

    for branch in ['PName', 'decayAncestor.PName', 'decayAncestor.Process', 'decayAncestor.VolName']:
        # Create a tree and add a branch 'EventNum' with the data array
        file["mczip1"] = uproot.newtree({branch: np.ndarray})  # Define the tree and branch structure
    
        # Write the data to the tree
        file["mczip1"].extend({branch: mczip[branch]})

AttributeError: module 'uproot' has no attribute 'newtree'

In [9]:
# Create a new ROOT file
root_file = ROOT.TFile('output_strings.root', 'RECREATE')

# Create a new tree in the ROOT file
tree = ROOT.TTree('mczip1', 'Tree for storing filtered data')

for branch in ['PName', 'decayAncestor.PName', 'decayAncestor.Process', 'decayAncestor.VolName']:

    # Create a string variable to hold the event name (ROOT uses C++ std::string)
    branch_data = ROOT.std.string()

    # Create a branch for the string data
    tree.Branch(branch, branch_data, branch+'/C')

    # Fill the tree with the string data
    for val in mczip[branch]:
        branch_data = val  # Assign the string value to the branch
        tree.Fill()

# Write the tree to the ROOT file
tree.Write()

# Close the ROOT file
root_file.Close()

In [10]:
mczipFrame = CDataFrame(f"mczip1", ['output_strings.root'])
mczipFrame.AsNumpy()

TypeError: function takes exactly 5 arguments (1 given)

In [12]:
f = ROOT.TFile(DMCfiles[0])
myTree = f.Get("G4SimDir/mczip1")
for entry in myTree:         
    PName = entry.PName

In [13]:
type(PName)

str